# **SMart Supermarket Customer**

This notebook analyzes the cleaned supermarket customer dataset to answer business questions related to customer value, product category spending, purchase channel behavior, campaign response, and retention priority.

In [ ]:
# Import Library
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)

## Load Data

In [ ]:
file_path = "supermarket_customers_cleaned.csv"

df = pd.read_csv(file_path)

df.head()

In [ ]:
#result 2240 rowa dan 29 column
df.shape

## Data Inspection

In [ ]:
#Dtype: object refer ke universal data spt string 
#untuk Dt_Customer Dtype seharusnya datetime bukan object
df.info()

In [ ]:
df['Dt_Customer'] = pd.to_datetime(df['Dt_Customer'])

df.info()

### Quick Validation

In [ ]:
#missing_values = df.isnull().sum().sort_values(ascending=False) or

missing_values = df.isna().sum()
missing_values = missing_values[missing_values > 0].sort_values(ascending=False)
missing_values

In [ ]:
df.duplicated().sum()

In [ ]:
df[['Age', 'Income', 'Recency', 'TotalSpend', 'TotalPurchases']].describe()

## Diagnostic Analysis: Supermarket Customer Behavior

Business Questions:

1. Which customers generate the highest total spending?
2. Which product categories contribute most to customer spending?
3. Which purchase channels are most used by customers?
4. What customer characteristics are associated with campaign response?
5. Which customers should be prioritized for retention?
6. How can the supermarket improve campaign targeting?

### Diagnostic Analysis 1: High-Value Customer Profile

**Business Question:** Which customers generate the highest total spending?

This analysis compares customer groups based on `CustomerValueSegment`. The goal is to understand what makes high-value customers different from mid-value and low-value customers.

#### 1.1 Customer Count by Value Segment

In [ ]:
value_segment_count = df['CustomerValueSegment'].value_counts().reset_index()
value_segment_count.columns = ['CustomerValueSegment', 'CustomerCount']

value_segment_count

#### 1.2 Spending Contribution by Value Segment

In [ ]:
value_segment_summary = df.groupby('CustomerValueSegment').agg(
    CustomerCount=('ID', 'count'),
    TotalSpend=('TotalSpend', 'sum'),
    AverageSpend=('TotalSpend', 'mean'),
    AverageIncome=('Income', 'mean'),
    AverageTotalPurchases=('TotalPurchases', 'mean'),
    AverageRecency=('Recency', 'mean'),
    AverageChildren=('Children', 'mean')
).reset_index()

value_segment_summary['SpendShare'] = (
    value_segment_summary['TotalSpend'] / value_segment_summary['TotalSpend'].sum()
)

value_segment_summary = value_segment_summary.sort_values(
    by='TotalSpend', ascending=False
)

value_segment_summary

#### 1.3 Top 20% Spending Contribution

In [ ]:
high_value = df[df['CustomerValueSegment'] == 'High Value']

top20_spend_share = high_value['TotalSpend'].sum() / df['TotalSpend'].sum()

print(f"Top 20% customers contribute {top20_spend_share:.2%} of total spending.")

In [ ]:
# Visual: Spending by Customer Value Segment

plt.figure(figsize=(8, 5))
plt.bar(value_segment_summary['CustomerValueSegment'], value_segment_summary['TotalSpend'])
plt.title('Total Spending by Customer Value Segment')
plt.xlabel('Customer Value Segment')
plt.ylabel('Total Spend')
plt.show()

In [ ]:
profile_summary = df.groupby('CustomerValueSegment').agg(
    AvgIncome=('Income', 'mean'),
    AvgTotalSpend=('TotalSpend', 'mean'),
    AvgTotalPurchases=('TotalPurchases', 'mean'),
    AvgRecency=('Recency', 'mean'),
    AvgChildren=('Children', 'mean'),
    AvgPreviousCampaignAccepted=('PreviousCampaignAcceptedTotal', 'mean')
).round(2)

profile_summary

### Diagnostic Analysis 2: Product Preference by Customer Segment

**Business Question:** Which product categories contribute most to customer spending?

This analysis identifies the strongest spending categories overall and compares product preferences across customer value segments.

#### 2.1 Total Spend by Product Category

In [ ]:
spending_cols = [
    'MntWines',
    'MntFruits',
    'MntMeatProducts',
    'MntFishProducts',
    'MntSweetProducts',
    'MntGoldProds'
]

product_spending = df[spending_cols].sum().sort_values(ascending=False).reset_index()
product_spending.columns = ['ProductCategory', 'TotalSpend']

product_spending

In [ ]:
#visual Product Category Spending
plt.figure(figsize=(9, 5))
plt.barh(product_spending['ProductCategory'], product_spending['TotalSpend'])
plt.title('Total Spending by Product Category')
plt.xlabel('Total Spend')
plt.ylabel('Product Category')
plt.gca().invert_yaxis()
plt.show()

#### 2.2 Product Spending by Customer Value Segment

In [ ]:
segment_product_spending = df.groupby('CustomerValueSegment')[spending_cols].sum().reset_index()

segment_product_spending

**Convert to Long Format for Tableau**

In [ ]:
segment_product_spending_long = segment_product_spending.melt(
    id_vars='CustomerValueSegment',
    value_vars=spending_cols,
    var_name='ProductCategory',
    value_name='TotalSpend'
)

segment_product_spending_long

#### 2.4 Average Product Spending by Segment

In [ ]:
segment_product_avg = df.groupby('CustomerValueSegment')[spending_cols].mean().round(2).reset_index()

segment_product_avg

### Diagnostic Analysis 3: Purchase Channel Behavior

**Business Question:** Which purchase channels are most used by customers?

This analysis compares total purchase counts across store, web, catalog, and deal channels.

#### 3.1 Total Purchases by Channel

In [ ]:
channel_cols = [
    'NumStorePurchases',
    'NumWebPurchases',
    'NumCatalogPurchases',
    'NumDealsPurchases'
]

channel_summary = df[channel_cols].sum().sort_values(ascending=False).reset_index()
channel_summary.columns = ['PurchaseChannel', 'TotalPurchases']

channel_summary

In [ ]:
#Visual: Purchase Channel Usage
plt.figure(figsize=(8, 5))
plt.bar(channel_summary['PurchaseChannel'], channel_summary['TotalPurchases'])
plt.title('Total Purchases by Channel')
plt.xlabel('Purchase Channel')
plt.ylabel('Total Purchases')
plt.xticks(rotation=30)
plt.show()

#### 3.2 Purchase Channel by Customer Value Segment

In [ ]:
segment_channel_summary = df.groupby('CustomerValueSegment')[channel_cols].sum().reset_index()

segment_channel_summary

**Convert to long format:**

In [ ]:
segment_channel_long = segment_channel_summary.melt(
    id_vars='CustomerValueSegment',
    value_vars=channel_cols,
    var_name='PurchaseChannel',
    value_name='TotalPurchases'
)

segment_channel_long

### Diagnostic Analysis 4: Campaign Response Drivers

**Business Question:** What customer characteristics are associated with campaign response?

This analysis compares customers who responded to the latest campaign with customers who did not respond.

#### 4.1 Campaign Response Rate

In [ ]:
response_summary = df['Response'].value_counts().reset_index()
response_summary.columns = ['Response', 'CustomerCount']

response_summary['ResponseLabel'] = response_summary['Response'].map({
    0: 'Did Not Respond',
    1: 'Responded'
})

response_summary['Percentage'] = response_summary['CustomerCount'] / response_summary['CustomerCount'].sum()

response_summary

#### 4.2 Responder vs Non-Responder Comparison

In [ ]:
response_comparison = df.groupby('Response').agg(
    CustomerCount=('ID', 'count'),
    AvgIncome=('Income', 'mean'),
    AvgTotalSpend=('TotalSpend', 'mean'),
    AvgTotalPurchases=('TotalPurchases', 'mean'),
    AvgRecency=('Recency', 'mean'),
    AvgChildren=('Children', 'mean'),
    AvgPreviousCampaignAccepted=('PreviousCampaignAcceptedTotal', 'mean')
).round(2)

response_comparison.index = response_comparison.index.map({
    0: 'Non-Responders',
    1: 'Responders'
})

response_comparison

In [ ]:
# Visual: Average Total Spend by Response
response_spend = df.groupby('Response')['TotalSpend'].mean().reset_index()
response_spend['ResponseLabel'] = response_spend['Response'].map({
    0: 'Non-Responders',
    1: 'Responders'
})

plt.figure(figsize=(6, 5))
plt.bar(response_spend['ResponseLabel'], response_spend['TotalSpend'])
plt.title('Average Total Spend by Campaign Response')
plt.xlabel('Campaign Response')
plt.ylabel('Average Total Spend')
plt.show()

In [ ]:
#Visual: Average Recency by Response
response_recency = df.groupby('Response')['Recency'].mean().reset_index()
response_recency['ResponseLabel'] = response_recency['Response'].map({
    0: 'Non-Responders',
    1: 'Responders'
})

plt.figure(figsize=(6, 5))
plt.bar(response_recency['ResponseLabel'], response_recency['Recency'])
plt.title('Average Recency by Campaign Response')
plt.xlabel('Campaign Response')
plt.ylabel('Average Recency, Days')
plt.show()

### Diagnostic Analysis 5: Retention Priority

**Business Question:** Which customers should be prioritized for retention?

This analysis identifies high-value customers who may be at risk because they have not purchased recently.

#### 5.1 Retention Segment Summary

In [ ]:
retention_summary = df.groupby('RetentionSegment').agg(
    CustomerCount=('ID', 'count'),
    TotalSpend=('TotalSpend', 'sum'),
    AverageSpend=('TotalSpend', 'mean'),
    AveragePurchases=('TotalPurchases', 'mean'),
    AverageRecency=('Recency', 'mean')
).reset_index()

retention_summary['SpendShare'] = retention_summary['TotalSpend'] / retention_summary['TotalSpend'].sum()

retention_summary = retention_summary.sort_values(by='TotalSpend', ascending=False)

retention_summary

In [ ]:
#Visual: Total Spend by Retention Segment
plt.figure(figsize=(9, 5))
plt.bar(retention_summary['RetentionSegment'], retention_summary['TotalSpend'])
plt.title('Total Spending by Retention Segment')
plt.xlabel('Retention Segment')
plt.ylabel('Total Spend')
plt.xticks(rotation=30)
plt.show()

#### 5.2 At-Risk High-Value Customer Summary

In [ ]:
at_risk_high_value = df[df['RetentionSegment'] == 'At-Risk High Value']

at_risk_summary = {
    'PriorityGroup': 'At-Risk High Value',
    'CustomerCount': at_risk_high_value['ID'].count(),
    'TotalSpend': at_risk_high_value['TotalSpend'].sum(),
    'SpendShare': at_risk_high_value['TotalSpend'].sum() / df['TotalSpend'].sum(),
    'AverageSpend': at_risk_high_value['TotalSpend'].mean(),
    'AverageRecency': at_risk_high_value['Recency'].mean(),
    'AverageTotalPurchases': at_risk_high_value['TotalPurchases'].mean()
}

at_risk_summary

**Convert to dataframe:**

In [ ]:
at_risk_summary_df = pd.DataFrame([at_risk_summary])
at_risk_summary_df

### Correlation Analysis

Correlation analysis is used to understand relationships between key numeric variables. This does not prove causation, but it helps identify useful patterns for segmentation and campaign targeting.

since this is right-skewed we will use spearman, mean > median

In [ ]:
corr_cols = [
    'Income',
    'TotalSpend',
    'TotalPurchases',
    'Recency',
    'Children',
    'PreviousCampaignAcceptedTotal',
    'Response'
]

spearman_corr = df[corr_cols].corr(method='spearman').round(2)

spearman_corr

In [ ]:


correlation_method_check = pd.DataFrame({
    'Mean': df[corr_cols].mean(),
    'Median': df[corr_cols].median(),
    'Min': df[corr_cols].min(),
    'Max': df[corr_cols].max(),
    'Skewness': df[corr_cols].skew(),
    'Data Type': df[corr_cols].dtypes
}).round(2)

correlation_method_check

def interpret_skewness(skew_value):
    if skew_value > 1:
        return 'Highly right-skewed'
    elif skew_value > 0.5:
        return 'Moderately right-skewed'
    elif skew_value < -1:
        return 'Highly left-skewed'
    elif skew_value < -0.5:
        return 'Moderately left-skewed'
    else:
        return 'Approximately symmetric'

correlation_method_check['Skewness Interpretation'] = correlation_method_check['Skewness'].apply(interpret_skewness)

correlation_method_check



In [ ]:
plt.figure(figsize=(8, 6))
plt.imshow(spearman_corr, aspect='auto')
plt.colorbar()

plt.xticks(
    range(len(spearman_corr.columns)), 
    spearman_corr.columns, 
    rotation=45, 
    ha='right'
)

plt.yticks(
    range(len(spearman_corr.index)), 
    spearman_corr.index
)

plt.title('Spearman Correlation Matrix of Key Variables')

for i in range(len(spearman_corr.index)):
    for j in range(len(spearman_corr.columns)):
        plt.text(
            j, 
            i, 
            spearman_corr.iloc[i, j], 
            ha='center', 
            va='center'
        )

plt.tight_layout()
plt.show()

### Export Diagnostic Summary Tables

In [ ]:
value_segment_summary.to_csv("diagnostics-csv-spearman/diagnostic_value_segment_summary.csv", index=False)
product_spending.to_csv("diagnostics-csv-spearman/diagnostic_product_spending_summary.csv", index=False)
segment_product_spending_long.to_csv("diagnostics-csv-spearman/diagnostic_segment_product_spending_long.csv", index=False)
channel_summary.to_csv("diagnostics-csv-spearman/diagnostic_channel_summary.csv", index=False)
segment_channel_long.to_csv("diagnostics-csv-spearman/diagnostic_segment_channel_long.csv", index=False)
response_summary.to_csv("diagnostics-csv-spearman/diagnostic_response_summary.csv", index=False)
response_comparison.to_csv("diagnostics-csv-spearman/diagnostic_response_comparison.csv")
retention_summary.to_csv("diagnostics-csv-spearman/diagnostic_retention_summary.csv", index=False)
at_risk_summary_df.to_csv("diagnostics-csv-spearman/diagnostic_at_risk_high_value_summary.csv", index=False)
spearman_corr.to_csv("diagnostics-csv-spearman/diagnostic_correlation_matrix.csv")

print("Diagnostic summary files exported successfully.")

### Business Question Answers

#### 1. Which customers generate the highest total spending?

High-value customers generate the highest total spending. They are defined as the top 20% of customers based on `TotalSpend`. These customers contribute a large share of total spending and should be prioritized for loyalty and retention actions.

#### 2. Which product categories contribute most to customer spending?

Wines and meat products contribute the most to total customer spending. These categories should be prioritized for personalized promotions and customer value campaigns.

#### 3. Which purchase channels are most used by customers?

Store purchases are the most used purchase channel, followed by web purchases. This suggests that the supermarket should maintain strong in-store engagement while also using digital campaigns to support omnichannel behavior.

#### 4. What customer characteristics are associated with campaign response?

Campaign responders tend to have stronger customer value and engagement signals, such as higher spending, more purchases, lower recency, and previous campaign acceptance. These customers are better candidates for targeted campaigns.

#### 5. Which customers should be prioritized for retention?

At-risk high-value customers should be prioritized. These are customers with high historical spending but higher recency, meaning they have not purchased recently. They should receive win-back offers and personalized retention campaigns.

#### 6. How can the supermarket improve campaign targeting?

The supermarket can improve campaign targeting by moving from broad mass campaigns to rule-based customer targeting.

Based on the analysis, future campaigns should prioritize customers with:

- High `TotalSpend`
- High `TotalPurchases`
- Low `Recency`
- Previous campaign acceptance
- Relevant product category preference
- High-value but inactive behavior

This approach helps the supermarket focus marketing effort on customers with stronger value, stronger engagement, and higher response potential.